# 09 — Advanced Experiment Designs

Standard A/B testing assumes you can randomly assign individual users to groups
and that their outcomes are independent. This assumption (SUTVA) breaks in three
common real-world situations:

| Situation | Example | SUTVA violation |
|---|---|---|
| Two-sided marketplace | DoorDash, Uber, Instacart | Treating a driver affects nearby riders |
| Ranking / recommendation | Netflix, Spotify | Can't show two algorithms to the same user at the same time |
| Social / network features | LinkedIn, Meta, Twitter | Treating one user changes their friends' experience |

Each requires a fundamentally different experimental design.

This notebook covers:

1. **Switchback experiments** — time-based randomization for marketplaces
2. **Interleaving** — simultaneous ranking comparison for recommenders
3. **Cluster / network randomization** — graph-based assignment for social features
4. **Decision framework** — when to use which design

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
np.random.seed(42)

---
## Design 1: Switchback Experiments
### The Problem: Marketplace Interference

On a two-sided marketplace (DoorDash, Lyft, Uber, Instacart), treating one side
of the market directly affects the other side. You cannot assign drivers to A/B groups
because drivers compete for the same orders — if half the drivers get a new incentive,
the remaining drivers' experience changes too. SUTVA is violated.

**Example:** DoorDash tests a new delivery incentive. If half the drivers in a city get
the incentive, they take more orders — leaving fewer orders for the control group drivers.
The control group's behaviour is contaminated by the treatment.

### The Solution: Time-Based Randomization

Instead of randomizing by user, randomize by **time window**:
- Odd hours → Treatment (new incentive active)
- Even hours → Control (no incentive)

Every driver is in control sometimes and treatment sometimes.
The unit of randomization is the (city, time_window) pair, not the individual driver.

### Key Design Decisions

| Decision | Consideration |
|---|---|
| Window length | Short windows → more periods, less power per period. Long → fewer periods, more spillover. |
| Washout period | Add a buffer between treatment and control windows to let spillover dissipate. |
| Randomization | Must be randomized (not alternating) to avoid systematic time-of-day confounding. |
| Analysis unit | Each (city, window) pair is one observation — NOT each driver or order. |

In [ ]:
# Simulate a switchback experiment: delivery platform testing a new incentive
N_DAYS    = 14
N_WINDOWS = N_DAYS * 24       # hourly windows
N_DRIVERS = 200               # active drivers per window
TRUE_LIFT = 0.08              # true treatment effect on completion rate
SPILLOVER = 0.03              # spillover effect on control windows from nearby treatment

# Randomly assign windows to treatment or control
treatment_mask = np.random.binomial(1, 0.5, N_WINDOWS).astype(bool)

# Simulate outcomes per window
base_rate = 0.70  # base completion rate
window_results = []

for w in range(N_WINDOWS):
    is_trt = treatment_mask[w]
    # Check if adjacent window was treatment (spillover)
    prev_trt = treatment_mask[w-1] if w > 0 else False

    if is_trt:
        rate = base_rate + TRUE_LIFT
    else:
        rate = base_rate + (SPILLOVER if prev_trt else 0)

    completions = np.random.binomial(N_DRIVERS, rate)
    window_results.append({
        "window": w,
        "treatment": int(is_trt),
        "completions": completions,
        "orders": N_DRIVERS,
        "rate": completions / N_DRIVERS,
    })

sw = pd.DataFrame(window_results)

# Naive analysis: t-test on window-level rates
trt_rates  = sw[sw["treatment"]==1]["rate"]
ctrl_rates = sw[sw["treatment"]==0]["rate"]
t_stat, p_val = stats.ttest_ind(trt_rates, ctrl_rates)
lift_obs = trt_rates.mean() - ctrl_rates.mean()

print(f"True lift:     {TRUE_LIFT:+.3f}")
print(f"Observed lift: {lift_obs:+.3f}")
print(f"p-value:       {p_val:.4f}")
print(f"N windows:     {len(sw)} ({sw.treatment.sum()} treatment, {(~sw.treatment.astype(bool)).sum()} control)")

In [ ]:
# Visualise: window-level outcomes and spillover effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: completion rates by window type
ax = axes[0]
ax.boxplot([ctrl_rates * 100, trt_rates * 100], labels=["Control windows", "Treatment windows"],
           patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.5))
ax.set_ylabel("Completion rate (%)")
ax.set_title("Distribution of completion rates by window type")

# Right: time series showing switchback pattern (first 72 hours)
ax = axes[1]
sw72 = sw.head(72)
colors = ["coral" if t else "steelblue" for t in sw72["treatment"]]
ax.bar(sw72["window"], sw72["rate"] * 100, color=colors, alpha=0.7, width=0.9)
ctrl_patch = mpatches.Patch(color="steelblue", alpha=0.7, label="Control window")
trt_patch  = mpatches.Patch(color="coral", alpha=0.7, label="Treatment window")
ax.legend(handles=[ctrl_patch, trt_patch])
ax.set_xlabel("Hour")
ax.set_ylabel("Completion rate (%)")
ax.set_title("Switchback pattern — first 72 hours")

plt.tight_layout()
plt.show()

print("Key challenge: spillover from treatment windows bleeds into adjacent control windows.")
print(f"Simulated spillover: {SPILLOVER:.0%} of true lift — biases control UP, shrinks observed lift.")

### Business Commentary

- **DoorDash** published extensively on switchback experiments for delivery incentive
  testing. Their platform can run switchbacks at the city, zone, or restaurant level.
- **Lyft** uses switchbacks for driver pay experiments and marketplace pricing.
- **Instacart** uses geo × time switchbacks for shopper incentive tests.

**The core trade-off:** Switchbacks get you valid causal estimates in marketplace
settings where user-level randomization is impossible — but they require many time
periods to accumulate statistical power, and spillover between windows must be
modelled or minimised (washout periods, longer window lengths).

**Variance challenge:** Each (city, window) is one independent observation.
With 14 days × 24 hours = 336 windows, you have 336 observations — not 336 × N_drivers.
Always cluster standard errors at the window level.

---
## Design 2: Interleaving
### The Problem: Ranking Experiments Are Expensive

To evaluate a new recommendation or search ranking algorithm with standard A/B testing,
you split users: Group A sees algorithm A's ranked list, Group B sees algorithm B's.
To detect a 1% improvement in CTR at 80% power, you might need millions of users
and weeks of runtime.

**Interleaving solves this by showing both algorithms to the same user simultaneously.**

### How It Works

1. Algorithm A produces a ranked list: [A1, A2, A3, A4, ...]
2. Algorithm B produces a ranked list: [B1, B2, B3, B4, ...]
3. Interleave them into a single list, alternating which algorithm goes first (randomised):
   - Team-draft interleaving: [A1, B1, B2, A2, A3, B3, ...]
4. Show the user the interleaved list. Track which algorithm's items they click.
5. Algorithm with more clicks wins.

### Why It's So Much More Sensitive

Each user simultaneously votes on both algorithms. In standard A/B, users are in
one group only — so between-user variance swamps the signal. Interleaving converts
between-user variance into within-user variance (which is much smaller).

**Netflix** reported interleaving is 100× more efficient than standard A/B for
evaluating recommendation quality. **Spotify** uses it for playlist and track ranking.

In [ ]:
# Simulate interleaving: comparing two recommendation algorithms
N_USERS = 10_000
LIST_LEN = 10           # items shown per user
TRUE_QUALITY_A = 0.12   # click prob for algorithm A's items
TRUE_QUALITY_B = 0.15   # click prob for algorithm B's items (B is 25% better)

# --- Standard A/B simulation ---
ab_results = []
for _ in range(N_USERS // 2):
    # Group A: see algorithm A only
    clicks_a = np.random.binomial(LIST_LEN, TRUE_QUALITY_A)
    ab_results.append(("A", clicks_a / LIST_LEN))
for _ in range(N_USERS // 2):
    # Group B: see algorithm B only
    clicks_b = np.random.binomial(LIST_LEN, TRUE_QUALITY_B)
    ab_results.append(("B", clicks_b / LIST_LEN))

ab_df = pd.DataFrame(ab_results, columns=["group", "ctr"])
ctr_a = ab_df[ab_df["group"]=="A"]["ctr"]
ctr_b = ab_df[ab_df["group"]=="B"]["ctr"]
t_ab, p_ab = stats.ttest_ind(ctr_b, ctr_a)

# --- Interleaving simulation ---
wins_b = 0  # times B got more clicks than A in same session
wins_a = 0
ties   = 0

for _ in range(N_USERS):
    # Each user sees LIST_LEN/2 items from each algorithm (interleaved)
    items_per_algo = LIST_LEN // 2
    clicks_a_items = np.random.binomial(items_per_algo, TRUE_QUALITY_A)
    clicks_b_items = np.random.binomial(items_per_algo, TRUE_QUALITY_B)
    if clicks_b_items > clicks_a_items:   wins_b += 1
    elif clicks_a_items > clicks_b_items: wins_a += 1
    else:                                  ties   += 1

n_decisive = wins_a + wins_b
b_win_rate = wins_b / n_decisive if n_decisive > 0 else 0.5
p_interleave = stats.binom_test(wins_b, n_decisive, 0.5)

print("STANDARD A/B:")
print(f"  CTR A: {ctr_a.mean():.4f} | CTR B: {ctr_b.mean():.4f}")
print(f"  p-value: {p_ab:.4f} | Significant: {p_ab < 0.05}")
print()
print("INTERLEAVING:")
print(f"  B wins: {wins_b:,} | A wins: {wins_a:,} | Ties: {ties:,}")
print(f"  B win rate: {b_win_rate:.3f}")
print(f"  p-value: {p_interleave:.6f} | Significant: {p_interleave < 0.05}")

In [ ]:
# Show: how many users needed to detect the same effect
from scipy.stats import norm

def ab_sample_size(p1, p2, alpha=0.05, power=0.80):
    p_bar = (p1 + p2) / 2
    z_a, z_b = norm.ppf(1 - alpha/2), norm.ppf(power)
    return int(np.ceil(2 * (z_a * np.sqrt(2*p_bar*(1-p_bar)) + z_b * np.sqrt(p1*(1-p1)+p2*(1-p2)))**2 / (p2-p1)**2))

n_ab = ab_sample_size(TRUE_QUALITY_A, TRUE_QUALITY_B)

# Interleaving: approximate using sign test on win/loss
# Win prob under H1: P(B wins a session) ≈ 0.5 + delta (depends on effect)
win_prob_h1 = b_win_rate  # empirical from simulation
n_interleave = ab_sample_size(0.5, win_prob_h1)

print(f"True quality: A={TRUE_QUALITY_A:.0%}, B={TRUE_QUALITY_B:.0%} (+{(TRUE_QUALITY_B-TRUE_QUALITY_A)/TRUE_QUALITY_A:.0%} relative)")
print(f"Standard A/B — users needed:    {n_ab:,}")
print(f"Interleaving  — users needed:    {n_interleave:,}")
print(f"Efficiency gain: {n_ab / max(n_interleave, 1):.1f}x fewer users")

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(["Standard A/B", "Interleaving"], [n_ab, n_interleave],
               color=["tomato", "steelblue"])
ax.set_xlabel("Users required to detect effect")
ax.set_title(f"Sample size comparison: A/B vs interleaving\n(detecting {TRUE_QUALITY_B/TRUE_QUALITY_A - 1:.0%} relative lift in ranking quality)")
for bar, val in zip(bars, [n_ab, n_interleave]):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

### Business Commentary

- **Netflix** uses interleaving to evaluate their recommendation algorithm changes.
  They reported needing 100× fewer users vs standard A/B for ranking evaluation.
- **Spotify** uses it for track and playlist ordering experiments.
- **Airbnb** uses interleaving for search ranking tests.

**Limitations:**
- Only works for ranking/ordering decisions — can't use for UI layout, pricing, etc.
- Position bias must be corrected (items at top get more clicks regardless of quality)
- Can't measure absolute metric levels — only relative preference between two algorithms
- Users seeing a blended list may behave differently than in production (ecological validity)

**Team-draft interleaving** (the most common method) handles position bias by giving
each algorithm the right to place its top-remaining item alternately. The algorithm
that goes first each round is randomised, controlling for position advantage.

---
## Design 3: Cluster / Network Randomization
### The Problem: Social Interference

When users interact with each other — liking posts, sending messages, seeing
"your friend also uses this" notifications — treating one user changes the experience
of their connections in the control group. This is **network interference**.

**Example:** LinkedIn tests a new "People You May Know" algorithm. If Alice is in
treatment, she gets better recommendations and connects with Bob. Bob is in control.
Bob's network just changed because of the treatment — he's contaminated.

### The Solution: Randomize by Network Cluster

1. Build the social graph (who is connected to whom)
2. Partition the graph into clusters (communities) using graph algorithms
3. Assign **entire clusters** to treatment or control
4. Users within a cluster only interact with others in the same cluster

**Trade-off:** Fewer independent units (clusters vs users) means lower statistical power.
The analysis must use cluster-robust standard errors.

In [ ]:
# Simulate a social network experiment with interference
N_USERS    = 2_000
N_CLUSTERS = 40      # graph communities
TRUE_LIFT  = 0.05    # true treatment effect on engagement
SPILLOVER  = 0.02    # how much treatment bleeds into connected control users

# Assign users to clusters (equal-size communities)
cluster_ids = np.repeat(np.arange(N_CLUSTERS), N_USERS // N_CLUSTERS)
cluster_treatment = np.random.binomial(1, 0.5, N_CLUSTERS)  # cluster-level assignment
user_treatment = cluster_treatment[cluster_ids]              # individual assignment

# Baseline engagement rate per cluster (cluster-level heterogeneity)
cluster_baselines = np.random.beta(5, 45, N_CLUSTERS)  # ~10% baseline, varies by cluster

# User outcomes: treatment effect + spillover
engagement = []
for i in range(N_USERS):
    c = cluster_ids[i]
    base = cluster_baselines[c]
    is_trt = user_treatment[i]
    # Spillover: control users in treatment-adjacent clusters get partial lift
    adjacent_trt = cluster_treatment[max(0, c-1)] or (cluster_treatment[min(N_CLUSTERS-1, c+1)])
    rate = base + (TRUE_LIFT if is_trt else (SPILLOVER if adjacent_trt else 0))
    engagement.append(np.random.binomial(1, min(rate, 1)))

net_df = pd.DataFrame({"cluster": cluster_ids, "treatment": user_treatment, "engaged": engagement})

# --- WRONG: naive user-level analysis (ignores clustering) ---
trt_e  = net_df[net_df["treatment"]==1]["engaged"]
ctrl_e = net_df[net_df["treatment"]==0]["engaged"]
lift_naive = trt_e.mean() - ctrl_e.mean()
t_naive, p_naive = stats.ttest_ind(trt_e, ctrl_e)

# --- CORRECT: cluster-level analysis (cluster-robust SE) ---
cluster_means = net_df.groupby(["cluster","treatment"])["engaged"].mean().reset_index()
trt_c  = cluster_means[cluster_means["treatment"]==1]["engaged"]
ctrl_c = cluster_means[cluster_means["treatment"]==0]["engaged"]
lift_cluster = trt_c.mean() - ctrl_c.mean()
t_cluster, p_cluster = stats.ttest_ind(trt_c, ctrl_c)

print(f"True lift:                {TRUE_LIFT:+.4f}")
print()
print(f"Naive user-level analysis:")
print(f"  Lift: {lift_naive:+.4f} | p: {p_naive:.4f} | N={len(net_df):,} users")
print(f"  -> Biased upward by spillover into control, SE too small (ignores clustering)")
print()
print(f"Cluster-level analysis (correct):")
print(f"  Lift: {lift_cluster:+.4f} | p: {p_cluster:.4f} | N={N_CLUSTERS} clusters")
print(f"  -> Correct estimate, wider CI (fewer independent units)")

In [ ]:
# Visualise: naive vs cluster analysis SE comparison
se_naive   = stats.sem(trt_e) + stats.sem(ctrl_e)
se_cluster = np.sqrt(np.var(trt_c, ddof=1)/len(trt_c) + np.var(ctrl_c, ddof=1)/len(ctrl_c))

fig, ax = plt.subplots(figsize=(8, 3.5))
methods = ["Naive\n(user-level)", "Cluster-robust\n(correct)"]
lifts  = [lift_naive, lift_cluster]
ses    = [se_naive,   se_cluster]
colors = ["tomato",   "steelblue"]

for i, (m, l, se, c) in enumerate(zip(methods, lifts, ses, colors)):
    z_c = stats.norm.ppf(0.975)
    ax.errorbar(l, i, xerr=[[z_c*se],[z_c*se]], fmt="o", color=c,
                capsize=10, markersize=10, linewidth=2.5)

ax.axvline(TRUE_LIFT, color="green", linewidth=2, linestyle="--", label=f"True lift ({TRUE_LIFT:+.3f})")
ax.axvline(0, color="gray", linewidth=1.2, linestyle=":")
ax.set_yticks([0, 1])
ax.set_yticklabels(methods)
ax.set_xlabel("Estimated lift (95% CI)")
ax.set_title("Network experiment: naive analysis is biased, cluster analysis is correct but wider")
ax.legend()
plt.tight_layout()
plt.show()

### Business Commentary

- **LinkedIn** randomizes by "ego network" for experiments on social feed and connections.
- **Meta** uses graph-cluster randomization for Facebook features that involve sharing,
  likes, or friend visibility.
- **Twitter/X** has published on network experiments for timeline ranking.

**The power cost is real:** going from N users to K clusters reduces your effective
sample size. With 2,000 users in 40 clusters, the analysis has 40 observations,
not 2,000. You need either more clusters or larger clusters to compensate.

**Variance estimation:** use cluster-robust standard errors (the Huber-White sandwich
estimator) or simply aggregate to the cluster level before running the t-test,
as shown above. Never run a user-level test when units are clustered.

---
## Decision Framework: Which Design to Use?

```
START: What is the treatment?
│
├── Ranking / ordering / recommendation?
│   └── → INTERLEAVING
│
├── Marketplace or supply-demand coupling?
│   └── Can you randomize by geography or time?
│       ├── Time-based → SWITCHBACK
│       └── Geo-based  → GEO HOLDOUT (similar principle)
│
├── Social / network feature?
│   └── Do users interact with each other?
│       ├── Yes → CLUSTER RANDOMIZATION
│       └── No  → STANDARD USER A/B
│
└── Individual product / UI / content feature?
    └── → STANDARD USER A/B
        └── Consider CUPED + Sequential Testing for efficiency
```

---

## Summary

| Method | Randomization unit | Key assumption broken | Efficiency vs standard A/B |
|---|---|---|---|
| Standard A/B | User | — | 1× |
| Switchback | Time window | Marketplace coupling | More periods needed |
| Interleaving | User (both algos) | Ranking requires choice | 10–100× more efficient |
| Cluster randomization | Social graph cluster | Network interference | Less efficient (fewer units) |

**The common thread:** all advanced designs exist because the independence assumption
of standard A/B testing is violated. The design choice always follows from the
structure of the interference — time, market, or network.

**Portfolio complete.** You now have a full picture from experiment design (notebook 01)
through to the methods used inside Meta, DoorDash, Netflix, and LinkedIn (notebooks 08–09).